# unbroadcast-pattern — worked example 2: Unbroadcast expanded size-1 axes

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `unbroadcast-pattern`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Broadcasting also expands axes that were size 1 in the original. To undo that, for each axis where `original.shape[i] == 1` but `grad.shape[i] != 1`, sum that axis with `keepdim=True` so the collapsed axis stays size 1 rather than disappearing.

## Worked solution

We undo broadcasting that expanded a size-1 axis.

1. The ranks already match here, so there are no leading axes to peel.
2. We scan each axis: where `original` had size 1 but `grad` is larger, that axis was broadcast.
3. `grad.sum(dim=i, keepdim=True)` sums across the expansion and keeps the axis at size 1, exactly matching `original`'s shape.

With `original` shape `(3, 1, 4)` and `grad` shape `(3, 5, 4)` of ones, axis 1 collapses back to size 1 and each surviving entry equals 5 — the number of expanded copies summed.

In [ ]:
import torch as t

def unbroadcast_size1(grad: t.Tensor, original: t.Tensor) -> t.Tensor:
    for i, size in enumerate(original.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

original = t.zeros(3, 1, 4)
grad = t.ones(3, 5, 4)
out = unbroadcast_size1(grad, original)
print('shape:', tuple(out.shape))
print('entry value:', out[0, 0, 0].item())